In [1]:
import numpy as np
import pandas as pd


df = pd.read_parquet('../Data/processed/features_final.parquet')
print(df.shape)

(307511, 183)


In [9]:
# Columnas categoricas
cat_cols = df.select_dtypes(include='str').columns.tolist()
print(f'Total de columnas categoricas: {len(cat_cols)}')
print(cat_cols)

Total de columnas categoricas: 16
['NAME_CONTRACT_TYPE', 'CODE_GENDER', 'FLAG_OWN_CAR', 'FLAG_OWN_REALTY', 'NAME_TYPE_SUITE', 'NAME_INCOME_TYPE', 'NAME_EDUCATION_TYPE', 'NAME_FAMILY_STATUS', 'NAME_HOUSING_TYPE', 'OCCUPATION_TYPE', 'WEEKDAY_APPR_PROCESS_START', 'ORGANIZATION_TYPE', 'FONDKAPREMONT_MODE', 'HOUSETYPE_MODE', 'WALLSMATERIAL_MODE', 'EMERGENCYSTATE_MODE']


In [11]:
for col in cat_cols:
    print(f'{col}: {df[col].nunique()} categorías')

NAME_CONTRACT_TYPE: 2 categorías
CODE_GENDER: 3 categorías
FLAG_OWN_CAR: 2 categorías
FLAG_OWN_REALTY: 2 categorías
NAME_TYPE_SUITE: 7 categorías
NAME_INCOME_TYPE: 8 categorías
NAME_EDUCATION_TYPE: 5 categorías
NAME_FAMILY_STATUS: 6 categorías
NAME_HOUSING_TYPE: 6 categorías
OCCUPATION_TYPE: 18 categorías
WEEKDAY_APPR_PROCESS_START: 7 categorías
ORGANIZATION_TYPE: 58 categorías
FONDKAPREMONT_MODE: 4 categorías
HOUSETYPE_MODE: 3 categorías
WALLSMATERIAL_MODE: 7 categorías
EMERGENCYSTATE_MODE: 2 categorías


Para las columnas con alta cardinalidad usaremos label encoding, que asigna un entero a cada categoría. Para el resto usaremos one-hot encoding que crea una columna binaria por cada categoria.

In [12]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()

for col in ['OCCUPATION_TYPE', 'ORGANIZATION_TYPE']:
    df[col] = le.fit_transform(df[col].astype('str'))

In [14]:
# Verificamos que la transformacion haya sido correcta
print(df[['OCCUPATION_TYPE', 'ORGANIZATION_TYPE']].dtypes)

OCCUPATION_TYPE      int64
ORGANIZATION_TYPE    int64
dtype: object


In [18]:
cat_cols_ohe = [c for c in df.select_dtypes(include='str').columns]

df = pd.get_dummies(df, columns=cat_cols_ohe, drop_first=True)

df.head()

,SK_ID_CURR,TARGET,CNT_CHILDREN,AMT_INCOME_TOTAL,AMT_CREDIT,AMT_ANNUITY,AMT_GOODS_PRICE,REGION_POPULATION_RELATIVE,DAYS_BIRTH,DAYS_EMPLOYED,...,FONDKAPREMONT_MODE_reg oper spec account,HOUSETYPE_MODE_specific housing,HOUSETYPE_MODE_terraced house,WALLSMATERIAL_MODE_Mixed,WALLSMATERIAL_MODE_Monolithic,WALLSMATERIAL_MODE_Others,WALLSMATERIAL_MODE_Panel,"WALLSMATERIAL_MODE_Stone, brick",WALLSMATERIAL_MODE_Wooden,EMERGENCYSTATE_MODE_Yes
0,100002,1,0,202500.0,406597.5,24700.5,351000.0,0.018801,-9461,-637.0,...,False,False,False,False,False,False,False,True,False,False
1,100003,0,0,270000.0,1293502.5,35698.5,1129500.0,0.003541,-16765,-1188.0,...,False,False,False,False,False,False,False,False,False,False
2,100004,0,0,67500.0,135000.0,6750.0,135000.0,0.010032,-19046,-225.0,...,False,False,False,False,False,False,True,False,False,False
3,100006,0,0,135000.0,312682.5,29686.5,297000.0,0.008019,-19005,-3039.0,...,False,False,False,False,False,False,True,False,False,False
4,100007,0,0,121500.0,513000.0,21865.5,513000.0,0.028663,-19932,-3038.0,...,False,False,False,False,False,False,True,False,False,False


In [20]:
# Separacion de datos en variables y target
X = df.copy().drop(['TARGET', 'SK_ID_CURR'], axis=1)
y = df['TARGET'].copy()

print(X.shape)
print(y.shape)

(307511, 217)
(307511,)


In [21]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.2, random_state=42, stratify=y)

print(X_train.shape)
print(X_test.shape)

(246008, 217)
(61503, 217)


In [24]:
from sklearn.preprocessing import StandardScaler

# Escalamos los datos
scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [25]:
from sklearn.dummy import DummyClassifier
from sklearn.metrics import roc_auc_score

dummy = DummyClassifier(strategy='most_frequent', random_state=42)
dummy.fit(X_train, y_train)

y_pred_dummy = dummy.predict(X_test)
y_prob_dummy = dummy.predict_proba(X_test)[:,1]

auc_dummy = roc_auc_score(y_test, y_prob_dummy)
print(f'Baseline AUC-ROC: {auc_dummy:.4f}')

Baseline AUC-ROC: 0.5000
